# Lezione 51 — I rischi del preference learning online

> **Modulo:** Feedback e preference training · **Tempo stimato:** 30 minuti
> **Prerequisiti:** Lezione 47 (reward), Lezione 49 (preference tuning).
>
> Obiettivo pratico unico: mostrare in NumPy il **reward hacking** (legge di
> Goodhart) — ottimizzare un proxy imperfetto fa salire il proxy ma **peggiorare**
> l'obiettivo vero. Chiude la Fase 7.

## Teoria essenziale

Il preference learning ha un lato oscuro, specie **online** (il modello impara di
continuo dal proprio comportamento):

- **Reward hacking / Goodhart**: si ottimizza un *proxy* (il reward model), non
  l'obiettivo vero. Se il proxy e' imperfetto, oltre un certo punto il modello
  impara a **sfruttarne i difetti**: il proxy sale, la qualita' vera scende.
- **Feedback loop**: gli output del modello diventano i dati di domani → i suoi
  bias si **amplificano** da soli.
- **Distribution shift**: le preferenze cambiano nel tempo; un modello congelato
  su vecchie preferenze si disallinea.

Mostriamo il primo, il piu' insidioso, con numeri.

In [1]:
import numpy as np

rng = np.random.default_rng(51)

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

d = 6
# direzione del PROXY: massimizzarlo spinge le feature lungo questa direzione
u = rng.normal(size=d)
u = u / np.linalg.norm(u)

# l'obiettivo VERO ha il suo ottimo in x_star. Lo costruiamo con una componente
# LUNGO u (cosi' all'inizio spingere sul proxy migliora anche il vero) e una
# ortogonale a u (che il proxy non vede). L'ascesa lungo u raggiunge l'ottimo
# vero e poi lo OLTREPASSA: da li' il vero cala mentre il proxy sale ancora.
_v = rng.normal(size=d)
_ortho = _v - (_v @ u) * u
_ortho = _ortho / np.linalg.norm(_ortho)
x_star = 4.5 * u + 3.0 * _ortho                  # proiezione 4.5 su u -> picco ~passo 30

def reward_proxy(x):
    return float(x @ u)                          # cresce spingendo lungo u

def qualita_vera(x):
    return float(-np.sum((x - x_star) ** 2))     # massima al target vero x_star

In [2]:
# una politica che massimizza il PROXY: gradient ascent lungo u
punto = np.zeros(d)
lr = 0.15
storia = []
for passo in range(60):
    punto = punto + lr * u                        # spinge sempre sul proxy
    storia.append((passo, reward_proxy(punto), qualita_vera(punto)))

for passo, rp, qv in storia[::12]:
    print(f"passo {passo:2d}: proxy {rp:7.2f}   qualita' vera {qv:8.2f}")

passo  0: proxy    0.15   qualita' vera   -27.92
passo 12: proxy    1.95   qualita' vera   -15.50
passo 24: proxy    3.75   qualita' vera    -9.56
passo 36: proxy    5.55   qualita' vera   -10.10
passo 48: proxy    7.35   qualita' vera   -17.12


Leggi le due colonne: il **proxy sale sempre**, ma la **qualita' vera** prima
**cresce** (migliorare il proxy migliora anche il vero) e poi, superato l'ottimo
vero, **cala** — la politica continua a spingere sul proxy oltrepassando cio' che
serve davvero. Ottimizzare troppo un proxy imperfetto danneggia l'obiettivo
reale.

In [3]:
proxy_vals = np.array([s[1] for s in storia])
vera_vals = np.array([s[2] for s in storia])

passo_max_vera = int(np.argmax(vera_vals))
passo_max_proxy = int(np.argmax(proxy_vals))
print(f"qualita' vera massima al passo {passo_max_vera}")
print(f"proxy massimo al passo       {passo_max_proxy}")
# il divario e' il cuore del problema di Goodhart
assert passo_max_proxy > passo_max_vera
print("-> il proxy continua a salire DOPO che la qualita' vera ha gia' iniziato "
      "a peggiorare: e' reward hacking")

qualita' vera massima al passo 29
proxy massimo al passo       59
-> il proxy continua a salire DOPO che la qualita' vera ha gia' iniziato a peggiorare: e' reward hacking


## Il progetto: Memory AI Lab, passo 51 — un freno di sicurezza

La difesa pratica: **early stopping** sull'obiettivo vero, misurato su un piccolo
campione umano tenuto da parte (un *hold-out* di preferenze fidate). Si smette di
ottimizzare quando la qualita' vera smette di salire, non quando il proxy e'
massimo.

In [4]:
def tuning_con_freno(storia, pazienza=5):
    migliore, passo_migliore = -np.inf, 0
    for passo, _proxy, vera in storia:
        if vera > migliore:
            migliore, passo_migliore = vera, passo
        elif passo - passo_migliore >= pazienza:
            return passo_migliore, migliore     # ferma: la vera non migliora piu'
    return passo_migliore, migliore

passo_stop, qual = tuning_con_freno(storia)
# il freno ferma vicino al massimo della qualita' vera, non a quello del proxy
assert passo_stop <= passo_max_proxy
assert abs(passo_stop - passo_max_vera) <= 5
print(f"early stopping sull'obiettivo vero: fermo al passo {passo_stop} "
      f"(qualita' {qual:.2f})")
print(f"(il proxy avrebbe continuato fino al passo {passo_max_proxy})")

early stopping sull'obiettivo vero: fermo al passo 29 (qualita' -9.00)
(il proxy avrebbe continuato fino al passo 59)


## Riepilogo (max 8 punti)

1. Il preference learning online ha rischi seri.
2. **Reward hacking / Goodhart**: si ottimizza un *proxy*, non l'obiettivo vero.
3. Oltre un punto, il proxy sale ma la **qualita' vera scende**.
4. **Feedback loop**: gli output diventano dati e amplificano i bias.
5. **Distribution shift**: le preferenze cambiano; un modello fisso si disallinea.
6. Il proxy massimo arriva **dopo** che la qualita' vera ha gia' iniziato a
   calare.
7. Difesa: **early stopping** sull'obiettivo vero, non sul proxy.
8. Serve un hold-out di preferenze umane fidate per misurarlo.

## Quiz

1. Cos'e' la legge di Goodhart, applicata al reward model?
2. Perche' un feedback loop puo' amplificare i bias?
3. Qual e' una difesa pratica contro il reward hacking?

*(Risposte: 1. quando un proxy diventa il bersaglio dell'ottimizzazione smette di
essere un buon proxy; 2. gli output del modello diventano i dati di training
successivi, rinforzando le sue stesse tendenze; 3. early stopping/validazione
sull'obiettivo vero misurato su un campione umano fidato.)*